# 🌊 Module 07: Generative Paradigms (Diffusion & Flow Matching)
### *From Thermodynamic Non-Equilibrium (2015) and DDPM (2020) to Optimal Transport Flow Matching (2023)*

Welcome to **Module 07 of Project Synapse**. In this notebook, we explore the generative modeling frontier that revolutionized modern AI (Stable Diffusion, Midjourney, Sora, Flux, and AlphaFold 3).

---

## 📚 Historical Evolution:
1. **2015 (Sohl-Dickstein et al.)**: *Deep Unsupervised Learning using Nonequilibrium Thermodynamics* — First proposed destroying structure via forward diffusion and reversing it.
2. **2020 (Ho, Jain, Abbeel)**: *Denoising Diffusion Probabilistic Models (DDPM)* — Connected diffusion to score matching (Song & Ermon) and parameterized networks to predict injected noise $\epsilon_\theta(x_t, t)$.
3. **2021 (Nichol & Dhariwal / Song et al.)**: Improved DDPM with cosine schedules and continuous score-based SDEs/ODEs.
4. **2023 (Lipman et al. / Albergo & Vanden-Eijnden)**: *Flow Matching & Stochastic Interpolants* — Dispensed with Brownian stochasticity in favor of straight Optimal Transport probability trajectories.

## 1. Mathematical Grounding: Forward Noise Schedules

The forward process corrupts data $ by injecting Gaussian noise according to a fixed variance schedule $\beta_1, \beta_2, \dots, \beta_T$:
17081q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1 - \beta_t} x_{t-1}, \beta_t \mathbf{I})17081

Using $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$, we sample at arbitrary step $ in closed form:
17081q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} x_0, (1 - \bar{\alpha}_t) \mathbf{I})17081
17081x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon, \quad \epsilon \sim \mathcal{N}(0, \mathbf{I})17081

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'python' / 'module07_generative_diffusion_flow'))

import math
import numpy as np
import matplotlib.pyplot as plt
import torch

from diffusion_ddpm_scratch import DiffusionNoiseSchedule, DenoisingMLP2D, DDPM2D, generate_toy_data
from flow_matching_toy import VelocityMLP2D, OptimalTransportFlowMatcher

print("PyTorch version:", torch.__version__)
print("All Module 07 dependencies loaded successfully!")

In [ ]:
# Compare Linear vs Cosine Variance Schedules
timesteps = 100
linear_sched = DiffusionNoiseSchedule(num_timesteps=timesteps, schedule_type='linear')
cosine_sched = DiffusionNoiseSchedule(num_timesteps=timesteps, schedule_type='cosine')

fig, axes = plt.subplots(1, 2, figsize=(14, 4), dpi=100)

axes[0].plot(linear_sched.betas.numpy(), label='Linear Beta Schedule', color='#06b6d4', lw=2)
axes[0].plot(cosine_sched.betas.numpy(), label='Cosine Beta Schedule', color='#a855f7', lw=2)
axes[0].set_title('Instantaneous Noise Variance $\beta_t$', fontsize=12)
axes[0].set_xlabel('Timestep $')
axes[0].set_ylabel('$\beta_t$')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(linear_sched.alphas_cumprod.numpy(), label='Linear $\bar{\alpha}_t$', color='#06b6d4', lw=2)
axes[1].plot(cosine_sched.alphas_cumprod.numpy(), label='Cosine $\bar{\alpha}_t$', color='#a855f7', lw=2)
axes[1].set_title('Cumulative Signal Retention $\bar{\alpha}_t$', fontsize=12)
axes[1].set_xlabel('Timestep $')
axes[1].set_ylabel('$\bar{\alpha}_t$')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Forward Gaussian Corruption on 2D Distributions

Watch how structured geometric data (Two Moons) gradually loses all topological information and dissolves into isotropic Gaussian noise as  \to T$.

In [ ]:
# Generate 2D Moons and apply forward corruption at multiple steps
data_x0 = generate_toy_data('two_moons', n_samples=2000)
tensor_x0 = torch.from_numpy(data_x0)

steps_to_show = [0, 20, 40, 70, 99]
fig, axes = plt.subplots(1, len(steps_to_show), figsize=(18, 3.5), dpi=100)

for idx, step in enumerate(steps_to_show):
    t_tensor = torch.full((len(tensor_x0),), step, dtype=torch.long)
    noisy_x, _ = linear_sched.q_sample(tensor_x0, t_tensor)
    pts = noisy_x.numpy()
    
    axes[idx].scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.5, color='#38bdf8')
    axes[idx].set_xlim(-3, 3)
    axes[idx].set_ylim(-3, 3)
    axes[idx].set_title(f' = {step}$ ($\bar{{\alpha}}_t = {linear_sched.alphas_cumprod[step]:.2f}$)')
    axes[idx].set_aspect('equal')
    axes[idx].grid(True, alpha=0.2)

plt.suptitle('Forward Diffusion: Information Dissolution  \to x_T$', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## 3. Training DDPM & Reverse Denoising Sampling

The reverse Markov chain transitions from pure Gaussian noise back into the data distribution:
17081x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z17081

In [ ]:
# Train 2D DDPM Model
model_ddpm = DenoisingMLP2D(hidden_dim=128, time_dim=64)
ddpm = DDPM2D(model_ddpm, linear_sched)
optimizer = torch.optim.AdamW(model_ddpm.parameters(), lr=3e-3)

print('Training DDPM model...')
for step in range(300):
    optimizer.zero_grad()
    loss = ddpm.compute_loss(tensor_x0)
    loss.backward()
    optimizer.step()
    if (step + 1) % 100 == 0:
        print(f'  Iteration {step + 1:3d} | Denoising MSE Loss: {loss.item():.4f}')

# Sample reverse denoising trajectory
samples, trajectory = ddpm.sample(num_samples=1000, return_trajectory=True)
print('Sampling completed! Trajectory recorded with shape:', trajectory.shape)

In [ ]:
# Plot reverse denoising snapshots from T down to 0
rev_steps = [100, 75, 50, 25, 0]
fig, axes = plt.subplots(1, len(rev_steps), figsize=(18, 3.5), dpi=100)

for idx, step in enumerate(rev_steps):
    step_idx = 100 - step
    pts = trajectory[step_idx]
    axes[idx].scatter(pts[:, 0], pts[:, 1], s=5, alpha=0.5, color='#ec4899')
    axes[idx].set_xlim(-3, 3)
    axes[idx].set_ylim(-3, 3)
    axes[idx].set_title(f'Reverse Step  = {step}$')
    axes[idx].set_aspect('equal')
    axes[idx].grid(True, alpha=0.2)

plt.suptitle('Reverse Diffusion: Structure Emergence  \to x_0$', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## 4. Optimal Transport Flow Matching (OT-CFM)

Flow Matching establishes straight, deterministic trajectories:
17081x_t = (1 - t) x_0 + t x_1, \quad u_t(x_t) = x_1 - x_017081
Objective:
17081\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t, x_0, x_1} \left[ \| v_\theta(x_t, t) - (x_1 - x_0) \|^2 \right]17081

In [ ]:
# Train OT-CFM Model
model_cfm = VelocityMLP2D(hidden_dim=128, time_dim=64)
cfm = OptimalTransportFlowMatcher(model_cfm)
optimizer_cfm = torch.optim.AdamW(model_cfm.parameters(), lr=3e-3)

print('Training OT-CFM...')
for step in range(300):
    optimizer_cfm.zero_grad()
    loss = cfm.compute_loss(tensor_x0)
    loss.backward()
    optimizer_cfm.step()
    if (step + 1) % 100 == 0:
        print(f'  Iteration {step + 1:3d} | CFM Velocity Loss: {loss.item():.4f}')

# Solve ODE using Midpoint RK2 in just 15 steps
cfm_samples, cfm_traj = cfm.solve_ode(num_samples=1000, num_steps=15, solver='midpoint')
print('OT-CFM ODE Integration complete!')

In [ ]:
# Vector Field Quiver & Straight Trajectories
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=100)

# 1. Learned Vector Field Quiver at t=0.5
X, Y, U, V = cfm.get_vector_field_grid(t_scalar=0.5, grid_res=16)
speed = np.sqrt(U**2 + V**2)
axes[0].quiver(X, Y, U, V, speed, cmap='cool', scale=25)
axes[0].set_title('Learned Velocity Vector Field \theta(x, t=0.5)$', fontsize=12)
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.2)

# 2. Straight ODE Trajectories vs DDPM Brownian Paths
for i in range(25):
    axes[1].plot(cfm_traj[:, i, 0], cfm_traj[:, i, 1], color='#10b981', alpha=0.4, lw=1.5)
axes[1].scatter(cfm_samples[:, 0], cfm_samples[:, 1], s=4, alpha=0.4, color='#059669', label='Generated Data')
axes[1].set_title('Straight OT-CFM Trajectories (Deterministic)', fontsize=12)
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.2)
axes[1].legend()

plt.tight_layout()
plt.show()